In [2]:
import numpy as np
import pickle
import time
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# -----------------------
# Load dataset
# -----------------------
def loadDataSet():
    fnSave = 'project2.dat'
    print(f'Loading data to {fnSave}')
    with open(fnSave, 'rb') as f:
        res = pickle.load(f)
    X_train = res['X_train']
    L_train = res['L_train'].astype(int)
    X_test = res['X_test']
    L_test = res['L_test'].astype(int)
    numCat = res['numCat']
    return X_train, L_train, X_test, L_test, numCat

# -----------------------
# Utilities
# -----------------------
def relu(z):
    return np.maximum(0, z)

# gradient of ReLU = step function
def relu_grad(z):
    return (z > 0).astype(float)

def softmax(z):
# divide by max for numerical stability and avoid numericaloverflow   
    z_shift = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def one_hot(labels, K):
# one-hot encoding of labels into a 2D binary matrix with K columns (number of classes) and rows corresponding to data points.
    Y = np.zeros((len(labels), K))
    Y[np.arange(len(labels)), labels] = 1.0
    return Y

def cross_entropy(Y, Q):
# Q is the 2D matrix of predicted probability distribution, Y is the one-hot encoded true labels    
    eps = 1e-12
    return -np.mean(np.sum(Y * np.log(Q + eps), axis=1))


In [ ]:
# -----------------------
# Shallow MLP
# -----------------------
class ShallowMLP:
    def __init__(self, input_dim=2, hidden_dim=8, output_dim=10, seed=None):
        rng = np.random.default_rng(seed)

        # He init to set the initial random weights
        self.W1 = rng.normal(0, np.sqrt(2 / input_dim), size=(hidden_dim, input_dim))
        self.b1 = np.zeros((1, hidden_dim))

        self.W2 = rng.normal(0, np.sqrt(2 / hidden_dim), size=(output_dim, hidden_dim))
        self.b2 = np.zeros((1, output_dim))

    def forward(self, X):
        # Z1 is the pre-activation of the hidden layer, A1 is the post-activation (after ReLU) 
        # Z2 is the pre-activation of the output layer, and Q is the final output probabilities after softmax.
        self.Z1 = X @ self.W1.T + self.b1           # (B, N)
        self.A1 = relu(self.Z1)                     # (B, N)
        self.Z2 = self.A1 @ self.W2.T + self.b2     # (B, K)
        self.Q = softmax(self.Z2)                   # (B, K)
        return self.Q

    def backward(self, X, Y):
        # B is the batch size
        B = X.shape[0]      

        # output error
        dZ2 = (self.Q - Y) / B                      # (B, K)

        # gradients for W2, b2
        dW2 = dZ2.T @ self.A1                       # (K, N)
        db2 = np.sum(dZ2, axis=0, keepdims=True)    # (1, K)

        # hidden error
        dZ1 = (dZ2 @ self.W2) * relu_grad(self.Z1)  # (B, N)

        # gradients for W1, b1
        dW1 = dZ1.T @ X                             # (N, 2)
        db1 = np.sum(dZ1, axis=0, keepdims=True)    # (1, N)

        return dW1, db1, dW2, db2

    def step(self, grads, lr):
        dW1, db1, dW2, db2 = grads
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2

    def predict(self, X):
        Q = self.forward(X)
        return np.argmax(Q, axis=1)


# Training Loop

In [ ]:
def train_model(X_train, L_train, X_test, L_test, numCat,
                hidden_dim=8, lr=0.01, epochs=500, batch_size=64, seed=None):

    model = ShallowMLP(input_dim=2, hidden_dim=hidden_dim, output_dim=numCat, seed=seed)

    Y_train = one_hot(L_train, numCat)
    losses = []

    n_train = len(X_train)

    for epoch in range(epochs):
        # shuffle each epoch
        idx = np.random.permutation(n_train)
        X_train_shuf = X_train[idx]
        Y_train_shuf = Y_train[idx]

        for start in range(0, n_train, batch_size):
            end = start + batch_size
            Xb = X_train_shuf[start:end]
            Yb = Y_train_shuf[start:end]

            model.forward(Xb)
            grads = model.backward(Xb, Yb)
            model.step(grads, lr)

        # monitor full training loss
        Q_train = model.forward(X_train)
        loss = cross_entropy(Y_train, Q_train)
        losses.append(loss)

    # final accuracies
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = np.mean(train_pred == L_train)
    test_acc = np.mean(test_pred == L_test)

    return model, losses, train_acc, test_acc, test_pred

In [ ]:
X_train, L_train, X_test, L_test, numCat = loadDataSet()

all_losses = []
all_test_acc = []
models = []
preds = []

for run in range(10):
    seed = int(time.time() * 1e6) % (2**32 - 1) + run
    model, losses, train_acc, test_acc, test_pred = train_model(
        X_train, L_train, X_test, L_test, numCat,
        hidden_dim=8, lr=0.01, epochs=500, batch_size=64, seed=seed
    )
    all_losses.append(losses)
    all_test_acc.append(test_acc)
    models.append(model)
    preds.append(test_pred)
    print(f"Run {run+1}: train_acc={train_acc:.4f}, test_acc={test_acc:.4f}")

# Plot Loss curve

In [ ]:
all_losses = np.array(all_losses)
mean_loss = all_losses.mean(axis=0)
std_loss = all_losses.std(axis=0)

plt.figure(figsize=(6,4))
plt.plot(mean_loss, label='mean training loss')
plt.fill_between(np.arange(len(mean_loss)),
                 mean_loss - std_loss,
                 mean_loss + std_loss,
                 alpha=0.3)
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss vs epoch (N=8)')
plt.legend()
plt.tight_layout()
plt.show()